# Tahoe Regional Transportation Plan Forecasting
> Data Engineering Tasks
* Residential development forecasting for 2035 and 2050

## Setup

In [1]:
# import packages
import pandas as pd
import pathlib
from pathlib import Path
import os
import arcpy
from utils import *
import numpy as np
import pickle
# external connection packages
from sqlalchemy.engine import URL
from sqlalchemy import create_engine

# pandas options
pd.options.mode.copy_on_write = True
pd.options.mode.chained_assignment = None
pd.options.display.max_columns = 999
pd.options.display.max_rows    = 999

# my workspace 
workspace = r"C:\GIS\Scratch.gdb"
# current working directory
local_path = pathlib.Path().absolute()

# get bonus_condit
# set data path as a subfolder of the current working directory TravelDemandModel\2022\
data_dir = local_path.parents[0] / 'data'
# folder to save processed data
out_dir  = local_path.parents[0] / 'data/processed_data'
# workspace gdb for stuff that doesnt work in memory
# gdb = os.path.join(local_path,'Workspace.gdb')
gdb = workspace
# set environement workspace to in memory 
arcpy.env.workspace = 'memory'
# # clear memory workspace
# arcpy.management.Delete('memory')

# overwrite true
arcpy.env.overwriteOutput = True
# Set spatial reference to NAD 1983 UTM Zone 10N
sr = arcpy.SpatialReference(26910)

# get parcels from the database
# network path to connection files
filePath = "F:/GIS/PARCELUPDATE/Workspace/"
# database file path 
sdeBase    = os.path.join(filePath, "Vector.sde")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")
sdeEdit    = os.path.join(filePath, "Edit.sde")

# Pickle variables
# part 1 - spatial joins and new categorical fields
parcel_pickle_part1    = data_dir / 'parcel_pickle1.pkl'
# part 2 - forecasting applied
parcel_pickle_part2    = data_dir / 'parcel_pickle2.pkl'

# columsn to list
initial_columns = [ 'APN',
                    'APO_ADDRESS',
                    'Residential_Units',
                    'TouristAccommodation_Units',
                    'CommercialFloorArea_SqFt',
                    'YEAR',
                    'JURISDICTION',
                    'COUNTY',
                    'OWNERSHIP_TYPE',
                    'COUNTY_LANDUSE_DESCRIPTION',
                    'EXISTING_LANDUSE',
                    'REGIONAL_LANDUSE',
                    'YEAR_BUILT',
                    'PLAN_ID',
                    'PLAN_NAME',
                    'ZONING_ID',
                    'ZONING_DESCRIPTION',
                    'TOWN_CENTER',
                    'LOCATION_TO_TOWNCENTER',
                    'TAZ',
                    'PARCEL_ACRES',
                    'PARCEL_SQFT',
                    'WITHIN_BONUSUNIT_BNDY',
                    'WITHIN_TRPA_BNDY',
                    'SHAPE']

### Functions

In [ ]:
# get SQL connection
def get_conn(db):
    # Get database user and password from environment variables on machine running script
    db_user             = os.environ.get('DB_USER')
    db_password         = os.environ.get('DB_PASSWORD')
    # driver is the ODBC driver for SQL Server
    driver              = 'ODBC Driver 17 for SQL Server'
    # server names are
    sql_12              = 'sql12'
    sql_14              = 'sql14'
    # make it case insensitive
    db = db.lower()
    # make sql database connection with pyodbc
    if db   == 'sde_tabular':
        connection_string = f"DRIVER={driver};SERVER={sql_12};DATABASE={db};UID={db_user};PWD={db_password}"
        connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
        engine = create_engine(connection_url)
    elif db == 'tahoebmpsde':
        connection_string = f"DRIVER={driver};SERVER={sql_14};DATABASE={db};UID={db_user};PWD={db_password}"
        connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
        engine = create_engine(connection_url)
    elif db == 'sde':
        connection_string = f"DRIVER={driver};SERVER={sql_12};DATABASE={db};UID={db_user};PWD={db_password}"
        connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
        engine = create_engine(connection_url)
    # else return None
    else:
        engine = None
    # connection file to use in pd.read_sql
    return engine

# save to pickle
def to_pickle(data, filename):
    with open(filename, 'wb') as f:
        pickle.dump(data, f)
    print(f'{filename} pickled')

# save to pickle and feature class
def to_pickle_fc(data, filename):
    data.spatial.to_featureclass(filename)
    with open(filename, 'wb') as f:
        pickle.dump(data, f)
    print(f'{filename} pickled and saved as feature class')

# get a pickled file as a dataframe
def from_pickle(filename):
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    print(f'{filename} unpickled')
    return data
def get_commercial_zones(df):
    columns_to_keep = ['Zoning_ID', 'Category', 'Density']
    # filter Use_Type to Multiple Family Dwelling
    df = df.loc[df['Category'] == 'Commercial']
    return df[columns_to_keep]

def get_tourist_zones(df):
    columns_to_keep = ['Zoning_ID', 'Category', 'Density']
    # filter Use_Type to Multiple Family Dwelling
    df = df.loc[df['Category'] == 'Tourist Accommodation']
    return df[columns_to_keep]

# function to get where Zoningin_ID Use_Type = Multi-Family and Density
def get_mf_zones(df):
    columns_to_keep = ['Zoning_ID', 'Use_Type', 'Density']
    # filter Use_Type to Multiple Family Dwelling
    df = df.loc[df['Use_Type'] == 'Multiple Family Dwelling']
    return df[columns_to_keep]

# function to get where Zoningin_ID Use_Type = Multi-Family and Density
def get_sf_zones(df):
    columns_to_keep = ['Zoning_ID', 'Use_Type', 'Density']
    # filter Use_Type to Multiple Family Dwelling
    df = df.loc[df['Use_Type'] == 'Single Family Dwelling']
    return df[columns_to_keep]

def get_mf_only_zones(df):
    columns_to_keep = ['Zoning_ID', 'Use_Type', 'Density']
    # filter Use_Type to Single Family Dwelling and not Multiple Family Dwelling
    dfMF = get_mf_zones(df)
    dfSF = get_sf_zones(df)
    # get Zoning_ID that are in both dataframes
    df = dfMF.loc[~dfMF['Zoning_ID'].isin(dfSF['Zoning_ID'])]
    return df[columns_to_keep]

def get_sf_only_zones(df):
    columns_to_keep = ['Zoning_ID', 'Use_Type', 'Density']
    # filter Use_Type to Single Family Dwelling and not Multiple Family Dwelling
    dfMF = get_mf_zones(df)
    dfSF = get_sf_zones(df)
    df = dfSF.loc[~dfSF['Zoning_ID'].isin(dfMF['Zoning_ID'])]
    return df[columns_to_keep]

def get_sf_mf_zones(df):
    columns_to_keep = ['Zoning_ID', 'Use_Type', 'Density']
    # get SF and MF zones
    dfSF = get_sf_zones(df)
    dfMF = get_mf_zones(df)
    # add the two dataframes together
    df = pd.concat([dfSF, dfMF])
    # only keep duplicate Zoning_ID
    df = df[df.duplicated(subset=['Zoning_ID'], keep=False)]
    return df[columns_to_keep]

def get_recieving_zones(df):
    columns_to_keep = ['Zoning_ID', 'SPECIAL_DESIGNATION']
    # filter transfer recieving
    df = df.loc[df['SPECIAL_DESIGNATION'] == 'Receive']
    return df[columns_to_keep]

def get_sending_zones(df):
    columns_to_keep = ['Zoning_ID', 'SPECIAL_DESIGNATION']
    df = df.loc[df['SPECIAL_DESIGNATION'] == 'Transfer']
    return df[columns_to_keep]


### Get Data

In [ ]:

# read in travel demand model base year parcel data
parcel_base_tdm  = r"C:\Users\amcclary\Documents\GitHub\Transportation\TravelDemandModel\2022\data\processed_data\sdf_parcel_part5.pkl"
parcel_base_path = local_path.parents[2].joinpath(parcel_base_tdm)
# read in base year parcel data
sdf_parcel_base  = pd.read_pickle(parcel_base_path)
output_gdb = r"C:\GIS\Scratch.gdb"
# parcel development layer polygons
parcel_db = Path(sdeEdit) / r"SDE.Parcel\\SDE.Parcel_History_Attributed"
# query 2022 rows
sdf_units = pd.DataFrame.spatial.from_featureclass(parcel_db)
sdf_units = sdf_units.loc[sdf_units['YEAR'] == 2022]
sdf_units.spatial.sr = sr

# # get parcel level data from Collection SDE
# vhr feature layer polygons 
vhr_db = Path(sdeCollect) / r"SDE.Parcel\\SDE.Parcel_VHR"
sdf_vhr = pd.DataFrame.spatial.from_featureclass(vhr_db)
sdf_vhr.spatial.sr = sr
# filter vhr layer to active status
sdf_vhr = sdf_vhr.loc[sdf_vhr['Status'] == 'Active']

# TAZ feature layer polygons
taz_db = Path(sdeBase) / r"SDE.Transportation\\SDE.Transportation_Analysis_Zone"
# get as spatial dataframe
sdf_taz = pd.DataFrame.spatial.from_featureclass(taz_db)
# set spatial reference to NAD 1983 UTM Zone 10N
sdf_taz.spatial.sr = sr

# censuse feature class
census_fc    = Path(sdeBase) / r"SDE.Census\\SDE.Tahoe_Census_Geography"
# bouns unit boundary feature class
bonus_unit_fc = Path(sdeBase) / r"SDE.Planning\\SDE.Bonus_unit_boundary"

# disable Z values on block group feature layer
with arcpy.EnvManager(outputZFlag="Disabled"):    
    arcpy.conversion.FeatureClassToGeodatabase(
        Input_Features="F:\GIS\DB_CONNECT\Vector.sde\SDE.Census\SDE.Tahoe_Census_Geography",
        Output_Geodatabase=output_gdb
    )
# disable Z values on block group feature layer
with arcpy.EnvManager(outputZFlag="Disabled"):    
    arcpy.conversion.FeatureClassToGeodatabase(
        Input_Features="F:\GIS\DB_CONNECT\Vector.sde\SDE.Planning\SDE.Bonus_unit_boundary",
        Output_Geodatabase=output_gdb
    )

# block group feature layer polygons with no Z
sdf_block = pd.DataFrame.spatial.from_featureclass(Path(gdb) / 'Tahoe_Census_Geography')
sdf_block = sdf_block.loc[(sdf_block['YEAR'] == 2020) & (sdf_block['GEOGRAPHY'] == 'Block Group')]
sdf_block.spatial.sr = sr

# bonus unit boundary wihtout Z
sdf_bonus = pd.DataFrame.spatial.from_featureclass(Path(gdb) / 'Bonus_unit_boundary')
sdf_bonus.spatial.sr = sr

# get parcel level data from LTinfo
dfIPES       = pd.read_json("https://www.laketahoeinfo.org/WebServices/GetParcelIPESScores/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476")
dfLCV_LTinfo = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetParcelsByLandCapability/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')
dfRetired    = pd.read_json("https://www.laketahoeinfo.org/WebServices/GetAllParcels/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476")
dfBankedDev  = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetBankedDevelopmentRights/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')
dfTransacted = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetTransactedAndBankedDevelopmentRights/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')
dfAllParcels = pd.read_json('https://www.laketahoeinfo.org/WebServices/GetAllParcels/JSON/e17aeb86-85e3-4260-83fd-a2b32501c476')

# get use tables 
# zoning data
sde_engine = get_conn('sde')
with sde_engine.begin() as conn:
    df_uses    = pd.read_sql("SELECT * FROM sde.SDE.PermissibleUses", conn)
    df_special = pd.read_sql("SELECT * FROM sde.SDE.Special_Designation", conn)

<>:32: SyntaxWarning: invalid escape sequence '\S'
<>:37: SyntaxWarning: invalid escape sequence '\G'
<>:43: SyntaxWarning: invalid escape sequence '\G'
<>:32: SyntaxWarning: invalid escape sequence '\S'
<>:37: SyntaxWarning: invalid escape sequence '\G'
<>:43: SyntaxWarning: invalid escape sequence '\G'
C:\Users\amcclary\AppData\Local\Temp\ipykernel_52672\451225646.py:32: SyntaxWarning: invalid escape sequence '\S'
  bonus_unit_fc = Path(sdeBase) / "SDE.Planning\SDE.Bonus_unit_boundary"
C:\Users\amcclary\AppData\Local\Temp\ipykernel_52672\451225646.py:37: SyntaxWarning: invalid escape sequence '\G'
  Input_Features="F:\GIS\DB_CONNECT\Vector.sde\SDE.Census\SDE.Tahoe_Census_Geography",
C:\Users\amcclary\AppData\Local\Temp\ipykernel_52672\451225646.py:43: SyntaxWarning: invalid escape sequence '\G'
  Input_Features="F:\GIS\DB_CONNECT\Vector.sde\SDE.Planning\SDE.Bonus_unit_boundary",
c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\arcgis\features\geo

### Parcel Data Engineering

In [ ]:
# Work from sdf_parcel_base since that's our current base year scenario
# This is just using the fully attributed parcel history layer to update fields in the sdf_parcel_base which is missing some data for the base year (2022)
update_fields = ['APO_ADDRESS', 'YEAR', 'COUNTY_LANDUSE_DESCRIPTION', 'REGIONAL_LANDUSE', 'YEAR_BUILT', 'PLAN_ID', 'PLAN_NAME', 'ZONING_ID', 'ZONING_DESCRIPTION', 'TOWN_CENTER', 'LOCATION_TO_TOWNCENTER', 'WITHIN_BONUSUNIT_BNDY']
for field in update_fields:
    sdf_parcel_base[field] = sdf_parcel_base.APN.map(dict(zip(sdf_units.APN, sdf_units[field]))) 

In [14]:

# # spatial join to get TAZ
# arcpy.SpatialJoin_analysis('sdf_units_fc', 'sdf_taz_fc', "Existing_Development_TAZ", 
#                            "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN")
# # spatial join to get Block Group
# arcpy.SpatialJoin_analysis('sdf_units_fc', 'sdf_block_fc', "Existing_Development_BlockGroup", 
#                            "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN")
# # spatial join of Bonus Unit Boundary


sdf_units_fc = 'memory\\sdf_units_fc'
sdf_bonus_fc = 'memory\\sdf_bonus_fc'
sdf_units.spatial.to_featureclass(sdf_units_fc)
sdf_bonus.spatial.to_featureclass(sdf_bonus_fc)
arcpy.SpatialJoin_analysis(sdf_units_fc, sdf_bonus_fc, "Existing_Development_BonusUnitBoundary",
                            "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "INTERSECT")

<Result 'memory\\Existing_Development_BonusUnitBoundary'>

In [20]:
# spatial dataframe with only initial columns
sdfParcels = sdf_parcel_base[initial_columns]

# get results of spatial joins as spatial dataframes
#sdf_units_taz   = pd.DataFrame.spatial.from_featureclass("Existing_Development_TAZ", sr=sr)  
#sdf_units_block = pd.DataFrame.spatial.from_featureclass("Existing_Development_BlockGroup", sr=sr)
sdf_units_bonus = pd.DataFrame.spatial.from_featureclass("Existing_Development_BonusUnitBoundary", sr=sr)
# cast to string
sdf_units_bonus['within_bonusunit_bndy'] = sdf_units_bonus['within_bonusunit_bndy'].astype(str)
sdf_units_bonus['within_bonusunit_bndy'] = 'No'
# if Id is not NA then within bonus unit boundary = yes, else
sdf_units_bonus.loc[sdf_units_bonus['id'].notna(), 'within_bonusunit_bndy'] = 'Yes'

# map dictionary to sdf_units dataframe to fill in TAZ and Block Group fields
#sdfParcels['TAZ']                   = sdfParcels.APN.map(dict(zip(sdf_units_taz.APN,   sdf_units_taz.TAZ_1)))
#sdfParcels['BLOCK_GROUP']           = sdfParcels.APN.map(dict(zip(sdf_units_block.APN, sdf_units_block.TRPAID)))
# map IPES score to parcels
sdfParcels['IPES_SCORE']            = sdfParcels['APN'].map(dict(zip(dfIPES.APN, dfIPES.IPESScore)))
sdfParcels['IPES_SCORE_TYPE']       = sdfParcels['APN'].map(dict(zip(dfIPES.APN, dfIPES.IPESScoreType)))
# retired parcels
sdfParcels['RETIRED']               = sdfParcels['APN'].map(dict(zip(dfAllParcels.APN, dfAllParcels.RetiredFromDevelopment)))
sdfParcels['WITHIN_BONUSUNIT_BNDY'] = sdfParcels['APN'].map(dict(zip(sdf_units_bonus.apn, sdf_units_bonus.within_bonusunit_bndy)))
# define housnig zoning and density
sdfParcels['HOUSING_ZONING']          = 'NA'
sdfParcels['COMMERCIAL_ALLOWED']      = 'No'
sdfParcels['TOURIST_ALLOWED']         = 'No'

# if the zoning id is in the list of multiple family zones then set to MF
sdfParcels.loc[sdfParcels['ZONING_ID'].isin(get_sf_mf_zones(df_uses)['Zoning_ID']), 'HOUSING_ZONING'] = 'SF/MF'
# if the zoning id is in the list of single family zones and not in the multiple family zones then set to SF only
sdfParcels.loc[sdfParcels['ZONING_ID'].isin(get_sf_only_zones(df_uses)['Zoning_ID']), 'HOUSING_ZONING'] = 'SF_only'
# if the zoning id is in the list of multiple family zones and not in the single family zones then set to MF only
sdfParcels.loc[sdfParcels['ZONING_ID'].isin(get_mf_only_zones(df_uses)['Zoning_ID']), 'HOUSING_ZONING'] = 'MF_only'
# if the zoning id is in the list of commercial zones then set to Commercial
sdfParcels.loc[sdfParcels['ZONING_ID'].isin(get_commercial_zones(df_uses)['Zoning_ID']), 'COMMERCIAL_ALLOWED'] = 'Yes'
# if the zoning id is in the list of tourist zones then set to Tourist Accommodation
sdfParcels.loc[sdfParcels['ZONING_ID'].isin(get_tourist_zones(df_uses)['Zoning_ID']), 'TOURIST_ALLOWED'] = 'Yes'

# if COUNTY is in EL or PL and SF allowed then set ADU_ALLOWED to yes or if COUNTY is in WA, DG, or CC and parcel acres is greater than 1 and SF allowed then set ADU_ALLOWED to yes
sdfParcels['ADU_ALLOWED'] = 'No'
sdfParcels.loc[(sdfParcels['COUNTY'].isin(['EL','PL', 'WA'])) & (~sdfParcels['HOUSING_ZONING'].isin(['MF_only', 'NA'])), 'ADU_ALLOWED'] = 'Yes'
sdfParcels.loc[(sdfParcels['COUNTY'].isin(['WA','DG','CC'])) & (sdfParcels['PARCEL_ACRES']>=1) &(~sdfParcels['HOUSING_ZONING'].isin(['MF_only', 'NA'])), 'ADU_ALLOWED'] = 'Yes'

# get density for MF and MF only zones, max residential units, and adjusted residential units
dfMF = get_mf_zones(df_uses)
sdfParcels['DENSITY']                    = sdfParcels['ZONING_ID'].map(dict(zip(dfMF.Zoning_ID, dfMF.Density)))
sdfParcels['MAX_RESIDENTIAL_UNITS']      = sdfParcels['PARCEL_ACRES'] * sdfParcels['DENSITY']
sdfParcels['MAX_UNITS']                  = sdfParcels['MAX_RESIDENTIAL_UNITS']*0.6
sdfParcels['MAX_UNITS']                  = sdfParcels['MAX_UNITS'].fillna(0).astype(int)

# set SF only zones to 1 max unit
sdfParcels.loc[sdfParcels['HOUSING_ZONING'] == 'SF_only', 'MAX_UNITS'] = 1

# set field for underbuilt evaluation
sdfParcels['POTENTIAL_UNITS'] = 0
sdfParcels['POTENTIAL_UNITS'] = sdfParcels['MAX_UNITS'] - sdfParcels['Residential_Units']
# set negative values to 0
sdfParcels.loc[sdfParcels['POTENTIAL_UNITS'] < 0, 'POTENTIAL_UNITS'] = 0

# calculate parcels with the greatest buildable potential  filter to the top 10% of parcels
# what value is in the top 10% of the potential buildable units
top_10_threshold = sdfParcels.POTENTIAL_UNITS.quantile(0.9)
# filter out rows where POTENTIAL_BUILDABLE_UNITS is NaN
sdfParcels['TOP_TEN_POTENTIAL_UNITS'] = sdfParcels.apply(lambda x: 'Yes' if x['POTENTIAL_UNITS'] >= top_10_threshold else 'No', axis=1)

# set FORECASTED_RESIDENTIAL_UNITS to 0
sdfParcels['FORECASTED_RESIDENTIAL_UNITS']     = 0
# set FORECAST_COMMERCIAN_UNITS to 0
sdfParcels['FORECASTED_COMMERCIAL_SQFT']       = 0
# set FORECAST_TOURIST_UNITS to 0
sdfParcels['FORECASTED_TOURIST_UNITS']         = 0
# set FORECAST_REASON to na
sdfParcels['FORECAST_REASON']                  = None
# FORECASTED_OCCUPANCY_RATE as a float field
sdfParcels['FORECASTED_RES_OCCUPANCY_RATE']    = 0.0

# export to pickle
sdfParcels.to_pickle(parcel_pickle_part1)
# to feature class
sdfParcels.spatial.to_featureclass(Path(gdb)/'Parcel_Base_2025')

'C:\\GIS\\Scratch.gdb\\Parcel_Base_2025'

## Model Year Forecasting

### Residential Forecasting

> Methods

1) Assign Known Projects - No longer relevant since this is from 2025
* Assign using Lookup_Lists\forecast_residential_assigned_units.csv: 
    * Known Residential Allocations from 2023-2024, 
    * Known Residential Bonus Units from permitted projects, 
    * applications in review, and 
    * RBU reservations in LT Info
    * Known Accessory Dwelling permits not completed
    * Remove Known Banking projects that removed units in 2023-2024
* Calcuate Remaining local jurisdiction pool units less units used for known projects
    
2) Assign Residential Bonus Units within Bonus Unit Boundary
* Full build out of CTC Asset Lands using jurisdiction bonus unit pools
* Identify vacant buildable lots within Bonus Unit boundary
* Assign remaining jurisdiction pool units to available parcels within jurisidiction

3) Assigne remaining Jurisdiction Residential Bonus and General Units
* Identify	Vacant buildable lots with allowed Multi-family use, calculate allowed density
* Assign 15% of remaining local jurisdiction Residential Allocation pool units to available multi-family parcels within jurisidiction
* Assign 35% of remaining Banked units to available multi-family parcels (use adjusted weighting from existing residential units?)
* Assign 35% of remaining Converted units to available multi-family parcels (use adjusted weighting from existing residential units?)
    
4) Assign remaining TRPA pool units to available parcels throughout region (use adjusted weighting from existing residential units?)
* Evaluate Vacant Buildable Lots with Single-family Residential Allowed Use	
* Identify	Vacant buildable lots with allowed Single-family use
* Identify	Accessory Dwelling Uses Allowed (All California Parcels and NV Parcels Greater than 1 Acre)
* Evaluate Underbuilt parcels with Multi-family Residential Allowed Use	
* Identify	Underbuilt Residential lots with allowed Multi-family use
* Evaluate Underbuilt parcels with Accessory Dwelling Uses Allowed (All California Parcels and NV Parcels Greater than 1 Acre)	
* Identify parcels that are in Town Centers or within a quarter mile and are top x% of underbuilt parcels. 
* Underbuilt = exclude existing condo and common area land uses. sf mf or mixed use
* ADU potential = exclude existing condo or common area land uses, and wait to use any NV parcels>acre. 


> Get the Parcel Base Data

In [21]:
# get pickle 1 as a spatially enabled dataframe - spatial joins, foreign keys, and new categorical fields added already
sdfParcels = from_pickle(parcel_pickle_part1)
# Randomly sort parcels so that development can be assigned randomly
sdfParcels = sdfParcels.sample(frac=1).reset_index(drop=True)
# Export index and apn to a csv file for future reference with a name that includes the date and time
# # This will allow us to recreate the same random order in the future
sdfParcels[['APN']].to_csv(f"Parcel_Sort_Order_{pd.Timestamp.now().strftime('%Y%m%d%H%M%S')}.csv", index=True)

c:\Users\amcclary\Documents\GitHub\Transportation\RegionalTransportationPlan\2023\data\parcel_pickle1.pkl unpickled


> Get Lookup Lists

In [22]:
# path to lookup lists
res_assigned_lookup = "Lookup_Lists/forecast_residential_assigned_units.csv"
res_zoned_lookup    = "Lookup_Lists/forecast_residential_zoned_units.csv"
ctc_assetlands_lookup = "Lookup_Lists/CTC_AssetLands_Lookup.csv"

# get zoned and assigned lookup lists as data frames
dfResZoned    = pd.read_csv(res_zoned_lookup)
dfResAssigned = pd.read_csv(res_assigned_lookup)
# get lookup list for CTC asset lands (17 parcels)
ctc_parcels = pd.read_csv(ctc_assetlands_lookup)

> Assign Known Projects

In [ ]:
# Forecast Known Residential Projects from 2023-2024
# group dfResAssigned by APN and sum Unit Change to aggregate to one total for duplciate 
# dfResAssignedGrouped_APN = dfResAssigned.groupby('APN').sum('Unit Change').reset_index()
# # assign forecast residential units for assigned projects
# sdfParcels['FORECASTED_RESIDENTIAL_UNITS'] = sdfParcels.APN.map(dict(zip(dfResAssignedGrouped_APN.APN, dfResAssignedGrouped_APN['Unit Change'])))
# # forecast reason = Assigned for APNs in dfResAssignedGrouped
# sdfParcels.loc[sdfParcels['APN'].isin(dfResAssigned['APN']), 'FORECAST_REASON'] = 'Assigned'

In [23]:
sdfParcels.groupby(['JURISDICTION','FORECAST_REASON'])['FORECASTED_RESIDENTIAL_UNITS'].sum()

Series([], Name: FORECASTED_RESIDENTIAL_UNITS, dtype: int64)

In [ ]:
# total forecasted units
total_forecasted_units = sdfParcels['FORECASTED_RESIDENTIAL_UNITS'].sum()
print(f'Total Forecasted Units: {total_forecasted_units}')

> Forecast full buildout of CTC Asset Lands

In [24]:
## CTC Asset Lands ##
# did we already do this?
# set the forecast reason to CTC Asset Lands for the 17 parcels n
sdfParcels.loc[(sdfParcels['APN'].isin(ctc_parcels['APN'])) & (sdfParcels['FORECAST_REASON']!='Assigned'), 'FORECAST_REASON'] = 'CTC Asset Lands'
# CTC asset lands that are truly buildable
ctc_condition          = "(sdfParcels['FORECAST_REASON'] == 'CTC Asset Lands') & (sdfParcels['MAX_UNITS'] > 0)"
# assign POTEINTIAL_UNITS to FORECASTED_RESIDENTIAL_UNITS for CTC Asset Lands that are buildable and not built by 2024
sdfParcels.loc[eval(ctc_condition), 'FORECASTED_RESIDENTIAL_UNITS'] = sdfParcels['MAX_UNITS']

In [25]:
sdfParcels.groupby(['JURISDICTION','FORECAST_REASON'])['FORECASTED_RESIDENTIAL_UNITS'].sum()

JURISDICTION  FORECAST_REASON
CSLT          CTC Asset Lands    233
EL            CTC Asset Lands     26
PL            CTC Asset Lands     13
Name: FORECASTED_RESIDENTIAL_UNITS, dtype: int64

### Add 900 VHRs to CSLT because of new regulation - Maybe move this to parcel setup

In [ ]:
import numpy as np


# South lake VHR conditions
VHR_condition = "(JURISDICTION == 'CSLT') & (Residential_Units > 0) & (VHR == 'No') & (TouristAccommodation_Units == 0)& (Residential_Units <3)"
CSLT_VHR_Eligble = sdfParcel.query(VHR_condition)
# Randomly assign New_VHR = 'Yes' to APNs until cumulative Residential_Units reaches 900


target_units = 900
shuffled = CSLT_VHR_Eligble.sample(frac=1, random_state=42).reset_index(drop=True)
cumulative = shuffled['Residential_Units'].cumsum()
selected_apns = shuffled.loc[cumulative <= target_units, 'APN']

sdfParcel['New_VHR'] = 'No'
sdfParcel.loc[sdfParcel['APN'].isin(selected_apns), 'New_VHR'] = 'Yes'

assigned_units = sdfParcel.loc[sdfParcel['New_VHR'] == 'Yes', 'Residential_Units'].sum()
print(f"Parcels assigned New_VHR = Yes: {sdfParcel['New_VHR'].eq('Yes').sum()}")
print(f"Total Residential_Units assigned: {assigned_units}")

In [ ]:
sdfParcel.loc[sdfParcel['New_VHR'] == 'Yes', 'VHR'] = 'Yes'
# if df.JURISDICTION == "CSLT" and VHR == "Yes" then set OCCUPANCY_ZONE to "CSLT_ALL"
sdfParcel.loc[(sdfParcel['JURISDICTION'] == 'CSLT') & (sdfParcel['VHR'] == 'Yes'), 'OCCUPANCY_ZONE'] = 'CSLT_ALL'

# columns to keep
sdfParcel = sdfParcel[final_schema]
# export to pickle
sdfParcel.to_pickle(parcel_pickle_part1)


> Subtract Assigned Units from the appropriate pool

In [26]:
# Subtract Assigned units from the appropriate pool
# group dfResAssigned by Jurisdiction and Unit_Pool
dfGroup_Assigned = dfResAssigned.groupby(['Jurisdiction', 'Unit_Pool']).sum('Unit Change').reset_index()
# drop Occupancy_Rate and Year columns
dfGroup_Assigned = dfGroup_Assigned.drop(columns=['Occupancy_Rate'])
# rename Unit Change to Unit_Change
dfGroup_Assigned = dfGroup_Assigned.rename(columns={'Unit Change':'Unit_Change'})

# merge dfResAssignedGrouped with dfResZoned on Jurisdiction and Pool
dfPool_Assigned = pd.merge(dfResZoned, dfGroup_Assigned, on=['Jurisdiction', 'Unit_Pool'], how='left')

# fill NaN with 0
dfPool_Assigned['Unit_Change'] = dfPool_Assigned['Unit_Change'].fillna(0)
# subtract known project aggregations from the zoned unit pools
dfPool_Assigned['Future_Units_Adjusted'] = dfPool_Assigned['Future_Units'] - dfPool_Assigned['Unit_Change']

> Subtract the CTC asset lands buildout from the appropriate pool

In [27]:
# filter to CTC Asset Lands parcels 
sdfCTC = sdfParcels.loc[sdfParcels['FORECAST_REASON'] == 'CTC Asset Lands']
# sum of forecasted residential units where forecast reason = ctc asset lands
sdfCTC_ForecastedByJurisdiction = sdfCTC.groupby('JURISDICTION')['FORECASTED_RESIDENTIAL_UNITS'].sum().reset_index()
# set unit pool to Bonus Unit or General based on Jurisdiction
sdfCTC_ForecastedByJurisdiction.loc[(sdfCTC_ForecastedByJurisdiction['JURISDICTION']=='CSLT'), 'Unit_Pool'] = 'Bonus Unit'
sdfCTC_ForecastedByJurisdiction.loc[(sdfCTC_ForecastedByJurisdiction['JURISDICTION']=='CSLT'), 'JURISDICTION']   = 'TRPA'

sdfCTC_ForecastedByJurisdiction.loc[(sdfCTC_ForecastedByJurisdiction['JURISDICTION']=='PL'), 'Unit_Pool']   = 'Bonus Unit'

sdfCTC_ForecastedByJurisdiction.loc[(sdfCTC_ForecastedByJurisdiction['JURISDICTION']=='EL'), 'Unit_Pool']   = 'Bonus Unit'
sdfCTC_ForecastedByJurisdiction.loc[(sdfCTC_ForecastedByJurisdiction['JURISDICTION']=='EL'), 'JURISDICTION'] = 'TRPA'

# mere rows with same Jurisdiction and Unit_Pool and sum forecasted residential units
sdfCTC_ForecastedByJurisdiction = sdfCTC_ForecastedByJurisdiction.groupby(['JURISDICTION', 'Unit_Pool'])['FORECASTED_RESIDENTIAL_UNITS'].sum().reset_index()

# rename columns
sdfCTC_ForecastedByJurisdiction.rename(columns={'JURISDICTION': 'Jurisdiction', 'FORECASTED_RESIDENTIAL_UNITS': 'Forecasted_CTC'}, inplace=True)

# Subtract CTC asset lands from the appropriate pool
dfPool_CTC = sdfCTC_ForecastedByJurisdiction.copy()

# merge with dfResMerge
dfPool_CTC = pd.merge(dfPool_Assigned, dfPool_CTC, on=['Jurisdiction', 'Unit_Pool'], how='left')

# fill NaN with 0
dfPool_CTC['Forecasted_CTC'] = dfPool_CTC['Forecasted_CTC'].fillna(0)
# subtract CTC asset lands from the appropriate pool
dfPool_CTC['Future_Units_Adjusted'] = dfPool_CTC['Future_Units_Adjusted'] - dfPool_CTC['Forecasted_CTC']
# add CTC Max Build to the unit change
dfPool_CTC['Unit_Change'] = dfPool_CTC['Unit_Change'] + dfPool_CTC['Forecasted_CTC']
# drop CTC Built column
dfPool_CTC.drop(columns=['Forecasted_CTC'], inplace=True)

# Start of scenarios

> Setup Unit Pools Proportion to be Used in Each Zone

In [28]:
dfPool = dfPool_CTC.copy()
# proportion target of forecast development by type
portion_multifamily = .35
portion_singlefamily = .5
portion_infill = .15
# Set units to use for each zoning type
dfPool['Future_Units_Adjusted'] = dfPool['Future_Units_Adjusted'].fillna(0)
dfPool['Future_Units_Adjusted_MF'] = (dfPool['Future_Units_Adjusted'] * portion_multifamily).round().astype(int)
dfPool['Future_Units_Adjusted_SF'] = (dfPool['Future_Units_Adjusted'] * portion_singlefamily).round().astype(int)
dfPool['Future_Units_Adjusted_Infill'] = (dfPool['Future_Units_Adjusted'] * portion_infill).round().astype(int)
# Assign any rounding error to the single family pool
dfPool['Adjustment'] = dfPool['Future_Units_Adjusted'] - dfPool['Future_Units_Adjusted_MF'] - dfPool['Future_Units_Adjusted_SF'] - dfPool['Future_Units_Adjusted_Infill']
dfPool['Future_Units_Adjusted_SF'] = dfPool['Future_Units_Adjusted_SF'] + dfPool['Adjustment']
dfPool.drop(columns=['Adjustment'], inplace=True)

# This is the part that will vary for different scenarios - move everything before this to a seperate notebook and have it export an updated unit balance csv

> Define Conditions

In [29]:
##------------------------------------ Conditional Statements for Forecasting Residential Unit Development------------------------------ ##
# vacant buildable criteria
vacant_buildable_criteria        = "(df['EXISTING_LANDUSE'] == 'Vacant') & (df['OWNERSHIP_TYPE'] == 'Private') & (df['RETIRED'] == 'No') & (df['IPES_SCORE'] > 0)"
placer_vacant_buildable_criteria = "(df['EXISTING_LANDUSE'] == 'Vacant') & (df['OWNERSHIP_TYPE'] == 'Private') & (df['RETIRED'] == 'No') & (df['IPES_SCORE'] > 726)" 

# Within TRPA Boundary as condition for all
trpa_boundary_criteria = "(df['WITHIN_TRPA_BNDY'] == 1)"
bonus_unit_criteria    = "(df['WITHIN_BONUSUNIT_BNDY'] == 'Yes')"
no_zoning_criteria     = "(df['HOUSING_ZONING'] != 'NA')"
sf_only_criteria       = "(df['HOUSING_ZONING'] == 'SF_only')"
mf_only_criteria       = "(df['HOUSING_ZONING'] == 'MF_only')"
sf_mf_criteria         = "(df['HOUSING_ZONING'].isin(['SF/MF', 'MF_only']))"
adu_criteria           = "(df['ADU_ALLOWED'] == 'Yes') & (df['Residential_Units']>0)"
towncenter_condition   = "(~df['TOWN_CENTER'].isna())"
top_10_condition       = "(df['TOP_TEN_POTENTIAL_UNITS'] == 'Yes')"
condo_size_condition   = "(df['PARCEL_ACRES'] >= 0.15)&(~df['EXISTING_LANDUSE'].isin(['Condominium', 'Condomunium Common Area']))"
ready_to_forecast      = "(df['FORECAST_REASON'].isna())&(df['OWNERSHIP_TYPE'] == 'Private')"

# setup f string to change jurisdiction in bonus condition
bonus_vacant_condition_template  = ("(df['JURISDICTION'] == '{}') & "  + 
                                    bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + 
                                    vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + 
                                    ready_to_forecast + " & " + condo_size_condition)
vacant_condition_template        = ("(df['JURISDICTION'] == '{}') & "  + 
                                    trpa_boundary_criteria + " & " + 
                                    vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + 
                                    ready_to_forecast + " & " + condo_size_condition)
placer_vacant_condition_template = ("(df['JURISDICTION'] == '{}') & " + 
                                    trpa_boundary_criteria + " & " + placer_vacant_buildable_criteria + " & " + 
                                    sf_mf_criteria + " & " + 
                                    ready_to_forecast + " & " + condo_size_condition)
infill_condition_template        = ("(df['JURISDICTION'] == '{}') & "  + 
                                    trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + 
                                    sf_mf_criteria + " & " + 
                                    ready_to_forecast + " & " + top_10_condition)
adu_condition_template           = ("(df['JURISDICTION'] == '{}') & "  + 
                                    trpa_boundary_criteria + " & " + adu_criteria + " & " + 
                                    ready_to_forecast)
towncenter_condition_template    = ("(df['JURISDICTION'] == '{}') & "  + 
                                    trpa_boundary_criteria + " & " + towncenter_condition + " & " + 
                                    ready_to_forecast)

# list of jurisdictions
jurisdictions = ['CSLT', 'DG', 'PL', 'WA','TRPA']
# loop through jurisdictions in bonus_condition
for j in jurisdictions:
    condition = bonus_vacant_condition_template.format(j)
    print(condition)

(df['JURISDICTION'] == 'CSLT') & (df['WITHIN_BONUSUNIT_BNDY'] == 'Yes') & (df['WITHIN_TRPA_BNDY'] == 1) & (df['EXISTING_LANDUSE'] == 'Vacant') & (df['OWNERSHIP_TYPE'] == 'Private') & (df['RETIRED'] == 'No') & (df['IPES_SCORE'] > 0) & (df['HOUSING_ZONING'].isin(['SF/MF', 'MF_only'])) & (df['FORECAST_REASON'].isna())&(df['OWNERSHIP_TYPE'] == 'Private') & (df['PARCEL_ACRES'] >= 0.15)&(~df['EXISTING_LANDUSE'].isin(['Condominium', 'Condomunium Common Area']))
(df['JURISDICTION'] == 'DG') & (df['WITHIN_BONUSUNIT_BNDY'] == 'Yes') & (df['WITHIN_TRPA_BNDY'] == 1) & (df['EXISTING_LANDUSE'] == 'Vacant') & (df['OWNERSHIP_TYPE'] == 'Private') & (df['RETIRED'] == 'No') & (df['IPES_SCORE'] > 0) & (df['HOUSING_ZONING'].isin(['SF/MF', 'MF_only'])) & (df['FORECAST_REASON'].isna())&(df['OWNERSHIP_TYPE'] == 'Private') & (df['PARCEL_ACRES'] >= 0.15)&(~df['EXISTING_LANDUSE'].isin(['Condominium', 'Condomunium Common Area']))
(df['JURISDICTION'] == 'PL') & (df['WITHIN_BONUSUNIT_BNDY'] == 'Yes') & (df['WITHIN_

In [ ]:
##------------------------------------ Conditional Statements for Forecasting Residential Unit Development------------------------------ ##
# vacant buildable criteria
vacant_buildable_criteria        = "(df['EXISTING_LANDUSE'] == 'Vacant') & (df['OWNERSHIP_TYPE'] == 'Private') & (df['RETIRED'] == 'No') & (df['IPES_SCORE'] > 0)"
placer_vacant_buildable_criteria = "(df['EXISTING_LANDUSE'] == 'Vacant') & (df['OWNERSHIP_TYPE'] == 'Private') & (df['RETIRED'] == 'No') & (df['IPES_SCORE'] > 726)" 

# Within TRPA Boundary as condition for all
trpa_boundary_criteria = "(df['WITHIN_TRPA_BNDY'] == 1)"
bonus_unit_criteria    = "(df['WITHIN_BONUSUNIT_BNDY'] == 'Yes')"
no_zoning_criteria     = "(df['HOUSING_ZONING'] != 'NA')"
sf_only_criteria       = "(df['HOUSING_ZONING'] == 'SF_only')"
mf_only_criteria       = "(df['HOUSING_ZONING'] == 'MF_only')"
sf_mf_criteria         = "(df['HOUSING_ZONING'].isin(['SF/MF', 'MF_only']))"
adu_criteria           = "(df['ADU_ALLOWED'] == 'Yes') & (df['Residential_Units']>0)"
towncenter_condition   = "(~df['TOWN_CENTER'].isna())"
top_10_condition       = "(df['TOP_TEN_POTENTIAL_UNITS'] == 'Yes')"
condo_size_condition   = "(df['PARCEL_ACRES'] >= 0.15)&(~df['EXISTING_LANDUSE'].isin(['Condominium', 'Condomunium Common Area']))"
ready_to_forecast      = "(df['FORECAST_REASON'].isna())&(df['OWNERSHIP_TYPE'] == 'Private')"

##------------------------------------------------- Jurisdiction Specific Conditions ---------------------------------------------------- ##

# jurisdiction bonus unit conditions
CSLT_Bonus_SF_condition       = "(df['JURISDICTION'] == 'CSLT')" + " & " + bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
CSLT_Bonus_MF_condition       = "(df['JURISDICTION'] == 'CSLT')" + " & " + bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
CSLT_Bonus_infill_condition   = "(df['JURISDICTION'] == 'CSLT')" + " & " + bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition

DG_Bonus_SF_condition         = "(df['JURISDICTION'] == 'DG')" + " & " + bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
DG_Bonus_MF_condition         = "(df['JURISDICTION'] == 'DG')" + " & " + bonus_unit_criteria + " & " +  trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
DG_Bonus_infill_condition     = "(df['JURISDICTION'] == 'DG')" + " & " + bonus_unit_criteria + " & " +  trpa_boundary_criteria +  " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition

PL_Bonus_SF_condition         = "(df['JURISDICTION'] == 'PL')" + " & " + bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + placer_vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
PL_Bonus_MF_condition         = "(df['JURISDICTION'] == 'PL')" + " & " + bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + placer_vacant_buildable_criteria + " & " + sf_only_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
PL_Bonus_infill_condition     = "(df['JURISDICTION'] == 'PL')" + " & " + bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition

WA_Bonus_MF_condition         = "(df['JURISDICTION'] == 'WA')" + " & " + bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_only_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
WA_Bonus_SF_condition         = "(df['JURISDICTION'] == 'WA')" + " & " + bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_only_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
WA_Bonus_infill_condition     = "(df['JURISDICTION'] == 'WA')" + " & " + bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition

# jurisdiction general conditions
CSLT_MF_condition             = "(df['JURISDICTION'] == 'CSLT') & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
CSLT_SF_condition             = "(df['JURISDICTION'] == 'CSLT') & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_only_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
CSLT_infill_condition         = "(df['JURISDICTION'] == 'CSLT') & " + trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition

DG_MF_condition               = "(df['JURISDICTION'] == 'DG') & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
DG_SF_condition               = "(df['JURISDICTION'] == 'DG') & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_only_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
DG_infill_condition           = "(df['JURISDICTION'] == 'DG') & " + trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition

EL_MF_condition               = "(df['JURISDICTION'] == 'EL') & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition  
EL_SF_condition               = "(df['JURISDICTION'] == 'EL') & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_only_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
EL_infill_condition           = "(df['JURISDICTION'] == 'EL') & " + trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition

PL_MF_condition               = "(df['JURISDICTION'] == 'PL') & " + trpa_boundary_criteria + " & " + placer_vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
PL_SF_condition               = "(df['JURISDICTION'] == 'PL') & " + trpa_boundary_criteria + " & " + placer_vacant_buildable_criteria + " & " + sf_only_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
PL_infill_condition           = "(df['JURISDICTION'] == 'PL') & " + trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition

WA_MF_condition               = "(df['JURISDICTION'] == 'WA') & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
WA_SF_condition               = "(df['JURISDICTION'] == 'WA') & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_only_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
WA_infill_condition           = "(df['JURISDICTION'] == 'WA') & " + trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition

# TRPA pool conditions
# bonus unit conditions
TRPA_Bonus_MF_condition       = bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
TRPA_Bonus_SF_condition       = bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_only_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
TRPA_Bonus_infill_condition   = bonus_unit_criteria + " & " + trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
# general conditions
TRPA_MF_condition             = trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
TRPA_SF_condition             = trpa_boundary_criteria + " & " + vacant_buildable_criteria + " & " + sf_only_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition
TRPA_infill_condition         = trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast+ " & " + condo_size_condition

# Town Center Pool Conditions
TC_MF_condition               = trpa_boundary_criteria + " & " + towncenter_condition + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
TC_SF_condition               = trpa_boundary_criteria + " & " + towncenter_condition + " & " + vacant_buildable_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
TC_infill_condition           = trpa_boundary_criteria + " & " + towncenter_condition + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition

# ADU Pool Conditions
TRPA_ADU_condition            = trpa_boundary_criteria + " & " + adu_criteria + " & " + ready_to_forecast + " & " + condo_size_condition

# Scenario 2 Conditions
# These will all be set to 0 in the main input files
TRPA_Affordable_condition       = trpa_boundary_criteria + " & " + towncenter_condition + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
TRPA_Moderate_condition         = trpa_boundary_criteria + " & " + bonus_unit_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
TRPA_Achievable_Bonus_condition         = trpa_boundary_criteria + " & " + bonus_unit_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
TRPA_Achievable_General_condition       = trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition
TRPA_Affordable_by_Design_condition       = trpa_boundary_criteria + " & " + sf_mf_criteria + " & " + ready_to_forecast + " & " + condo_size_condition


In [31]:
conditions = {
                'CSLT_Bonus_SF'      : CSLT_Bonus_SF_condition, 
                'CSLT_Bonus_MF'      : CSLT_Bonus_MF_condition, 
                'CSLT_Bonus_Infill'  : CSLT_Bonus_infill_condition,
                'DG_Bonus_SF'        : DG_Bonus_SF_condition,
                'DG_Bonus_MF'        : DG_Bonus_MF_condition,
                'DG_Bonus_Infill'    : DG_Bonus_infill_condition,
                'PL_Bonus_SF'        : PL_Bonus_SF_condition,
                'PL_Bonus_MF'        : PL_Bonus_MF_condition,
                'PL_Bonus_Infill'    : PL_Bonus_infill_condition,
                'WA_Bonus_SF'        : WA_Bonus_SF_condition,
                'WA_Bonus_MF'        : WA_Bonus_MF_condition,
                'WA_Bonus_Infil'     : WA_Bonus_infill_condition,
                'CSLT_General_MF'    : CSLT_MF_condition,
                'CSLT_General_SF'    : CSLT_SF_condition,
                'CSLT_General_Infill': CSLT_infill_condition,
                'DG_General_MF'      : DG_MF_condition,
                'DG_General_SF'      : DG_SF_condition,
                'DG_General_Infill'  : DG_infill_condition,
                'EL_General_MF'      : EL_MF_condition,
                'EL_General_SF'      : EL_SF_condition,
                'EL_General_Infill'  : EL_infill_condition,
                'PL_General_MF'      : PL_MF_condition,
                'PL_General_SF'      : PL_SF_condition,
                'PL_General_Infill'  : PL_infill_condition,
                'WA_General_MF'      : WA_MF_condition,
                'WA_General_SF'      : WA_SF_condition,
                'WA_General_Infill'  : WA_infill_condition,
                'TRPA_Bonus_MF'      : TRPA_Bonus_MF_condition,
                'TRPA_Bonus_SF'      : TRPA_Bonus_SF_condition,
                'TRPA_Bonus_Infill'  : TRPA_Bonus_infill_condition,
                'TRPA_General_MF'    : TRPA_MF_condition,
                'TRPA_General_SF'    : TRPA_SF_condition,
                'TRPA_General_Infill': TRPA_infill_condition,
                'TRPA_ADU'           : adu_criteria,
                'TC_MF'              : TC_MF_condition,
                'TC_SF'              : TC_SF_condition,
                'TC_Infill'          : TC_infill_condition,
                }

In [32]:
# check parcel conditions
df_potential = pd.DataFrame()
for key,value in conditions.items():
    df = check_parcel_condition(sdfParcels, value)
    df['Condition'] = key
    df_potential = pd.concat([df_potential, df], axis=0)

df_potential

,Parcels_Available,Total_Max_Units,Total_Potential_Units,Total_Existing_Units,Condition
0,96,408,399,9,CSLT_Bonus_SF
0,96,408,399,9,CSLT_Bonus_MF
0,2052,37318,35097,3875,CSLT_Bonus_Infill
0,4,40,40,0,DG_Bonus_SF
0,4,40,40,0,DG_Bonus_MF
0,451,5827,5433,578,DG_Bonus_Infill
0,41,156,156,1,PL_Bonus_SF
0,224,224,222,2,PL_Bonus_MF
0,1148,6428,5553,1161,PL_Bonus_Infill
0,53,53,52,1,WA_Bonus_SF


> Forecast Jurisdiction Pools

In [34]:
## Jurisdictional Forecasting ##
##-----------------------------------------------------------Bonus Unit Assignments-----------------------------------------------------------##
## CSLT Bonus Unit Assignments ##
# Assign Bonus Units to Multifamily zoned parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'CSLT', 'Bonus Unit', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, CSLT_Bonus_MF_condition, target_sum, 'CSLT Bonus Units MF')
df_built_parcels = df_summary
# Assign Bonus Units to Single Family only zoned parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'CSLT', 'Bonus Unit', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, CSLT_Bonus_SF_condition, target_sum, 'CSLT Bonus Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign Bonus Units to Developable Infill parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'CSLT', 'Bonus Unit', 'Infill')
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, CSLT_Bonus_infill_condition, target_sum, 'CSLT Bonus Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

## Douglas Bonus Unit Assignments ##
# Assign Bonus Units to Multifamily zoned parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'DG', 'Bonus Unit', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, DG_Bonus_MF_condition, target_sum, 'DG Bonus Units MF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign Bonus Units to Single Family only zoned parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'DG', 'Bonus Unit', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, DG_Bonus_SF_condition, target_sum, 'DG Bonus Units SF')
# Assign Bonus Units to Developable Infill parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'DG', 'Bonus Unit', 'Infill')
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, DG_Bonus_infill_condition, target_sum, 'DG Bonus Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

## Placer Bonus Unit Assignments ##
# Assign Bonus Units to Multifamily zoned parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'PL', 'Bonus Unit', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, PL_Bonus_MF_condition, target_sum, 'PL Bonus Units MF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign Bonus Units to Single Family only zoned parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'PL', 'Bonus Unit', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, PL_Bonus_SF_condition, target_sum, 'PL Bonus Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign Bonus Units to Developable Infill parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'PL', 'Bonus Unit', 'Infill')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, PL_Bonus_infill_condition, target_sum, 'PL Bonus Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

## Washoe Bonus Unit Assignments ##
# Assign Bonus Units to Multifamily zoned parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'WA', 'Bonus Unit', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, WA_Bonus_MF_condition, target_sum, 'WA Bonus Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign Bonus Units to Single Family only zoned parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'WA', 'Bonus Unit', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, WA_Bonus_SF_condition, target_sum, 'WA Bonus Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign Bonus Units to Developable Infill parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'WA', 'Bonus Unit', 'Infill')
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, WA_Bonus_infill_condition, target_sum, 'WA Bonus Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

## ----------------------------------------------------General Unit Assignments---------------------------------------------------- ##
## CSTL General Unit Assignments ##
# Assign General Jurisdiction Units to Multifamily zoned parcels
target_sum             = get_target_sum(dfPool, 'CSLT', 'General', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, CSLT_MF_condition, target_sum, 'CSLT General Units MF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Single Family only zoned parcels
target_sum             = get_target_sum(dfPool, 'CSLT', 'General', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, CSLT_SF_condition, target_sum, 'CSLT General Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Developable Infill parcels
target_sum             = get_target_sum(dfPool, 'CSLT', 'General', 'Infill')
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, CSLT_infill_condition, target_sum, 'CSLT General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

## El Dorado General Unit Assignments ##
# Assign General Jurisdiction Units to Multifamily zoned parcels
target_sum             = get_target_sum(dfPool, 'EL', 'General', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, EL_MF_condition, target_sum, 'EL General Units MF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Single Family only zoned parcels
target_sum             = get_target_sum(dfPool, 'EL', 'General', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, EL_SF_condition, target_sum, 'EL General Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Developable Infill parcels
target_sum             = get_target_sum(dfPool, 'EL', 'General', 'Infill')
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, EL_infill_condition, target_sum, 'EL General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

# Placer General Unit Assignments
# Assign General Jurisdiction Units to Multifamily zoned parcels
target_sum             = get_target_sum(dfPool, 'PL', 'General', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, PL_MF_condition, target_sum, 'PL General Units MF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Single Family only zoned parcels
target_sum             = get_target_sum(dfPool, 'PL', 'General', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, PL_SF_condition, target_sum, 'PL General Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Developable Infill parcels
target_sum             = get_target_sum(dfPool, 'PL', 'General', 'Infill')
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, PL_infill_condition, target_sum, 'PL General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

# Douglas General Unit Assignments
# Assign General Jurisdiction Units to Multifamily zoned parcels
target_sum             = get_target_sum(dfPool, 'DG', 'General', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, DG_MF_condition, target_sum, 'DG General Units MF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Single Family only zoned parcels
target_sum             = get_target_sum(dfPool, 'DG', 'General', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, DG_SF_condition, target_sum, 'DG General Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Developable Infill parcels
target_sum             = get_target_sum(dfPool, 'DG', 'General', 'Infill')
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, DG_infill_condition, target_sum, 'DG General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

# Washoe General Unit Assignments
# Assign General Jurisdiction Units to Multifamily zoned parcels 
target_sum             = get_target_sum(dfPool, 'WA', 'General', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, WA_MF_condition, target_sum, 'WA General Units MF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Single Family only zoned parcels
target_sum             = get_target_sum(dfPool, 'WA', 'General', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, WA_SF_condition, target_sum, 'WA General Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Developable Infill parcels
target_sum             = get_target_sum(dfPool, 'WA', 'General', 'Infill')
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, WA_infill_condition, target_sum, 'WA General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

df_built_parcels

,Reason,Parcels_Available,Parcels_Used,Total_Forecasted_Units,Total_Remaining_Units
0,CSLT Bonus Units MF,53,10,24.0,0.0
1,CSLT Bonus Units SF,43,5,34.0,0.0
2,CSLT Bonus Units Infill,1993,2,10.0,0.0
3,DG Bonus Units MF,0,0,0.0,23.0
4,DG Bonus Units Infill,445,3,10.0,0.0
5,PL Bonus Units MF,214,10,10.0,0.0
6,PL Bonus Units SF,37,2,14.0,0.0
7,PL Bonus Units Infill,1140,2,4.0,0.0
8,WA Bonus Units SF,0,0,0.0,42.0
9,WA Bonus Units SF,0,0,0.0,60.0


> Forecast TRPA Pools

In [35]:
## TRPA Unit Pool Assignments ##
##-----------------------------------------------------------Bonus Unit Assignments-----------------------------------------------------------##
## TRPA Bonus Unit Assignments ##
# Assign Bonus Units to Multifamily zoned parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'TRPA', 'Bonus Unit', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, TRPA_Bonus_MF_condition, target_sum, 'TRPA Bonus Units MF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign Bonus Units to Single Family only zoned parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'TRPA', 'Bonus Unit', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, TRPA_Bonus_SF_condition, target_sum, 'TRPA Bonus Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign Bonus Units to Developable Infill parcels within the Bonus Unit Boundary
target_sum             = get_target_sum(dfPool, 'TRPA', 'Bonus Unit', 'Infill')
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, TRPA_Bonus_infill_condition, target_sum, 'TRPA Bonus Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

## ---------------------------------------------------------TRPA General Unit Assignments--------------------------------------------------------- ##
# Assign General Jurisdiction Units to Multifamily zoned parcels
target_sum             = get_target_sum(dfPool, 'TRPA', 'General', 'MF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, TRPA_MF_condition, target_sum, 'TRPA General Units MF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Single Family only zoned parcels
target_sum             = get_target_sum(dfPool, 'TRPA', 'General', 'SF')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, TRPA_SF_condition, target_sum, 'TRPA General Units SF')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)
# Assign General Jurisdiction Units to Developable Infill parcels
target_sum             = get_target_sum(dfPool, 'TRPA', 'General', 'Infill')
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, TRPA_infill_condition, target_sum, 'TRPA General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

## TRPA ADU Unit Assignments ##
# Assign ADU Units to parcels
target_sum             = get_target_sum(dfPool, 'TRPA', 'ADU', 'ADU')
sdfParcels, df_summary = forecast_residential_units(sdfParcels, TRPA_ADU_condition, target_sum, 'TRPA ADU Units')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

# summarize total units forecasted
df_built_parcels

,Reason,Parcels_Available,Parcels_Used,Total_Forecasted_Units,Total_Remaining_Units
0,CSLT Bonus Units MF,53,10,24.0,0.0
1,CSLT Bonus Units SF,43,5,34.0,0.0
2,CSLT Bonus Units Infill,1993,2,10.0,0.0
3,DG Bonus Units MF,0,0,0.0,23.0
4,DG Bonus Units Infill,445,3,10.0,0.0
5,PL Bonus Units MF,214,10,10.0,0.0
6,PL Bonus Units SF,37,2,14.0,0.0
7,PL Bonus Units Infill,1140,2,4.0,0.0
8,WA Bonus Units SF,0,0,0.0,42.0
9,WA Bonus Units SF,0,0,0.0,60.0


> Assign the Remainders

In [36]:
# use df_built_parcels to set new infill target_sum
df_remaining = df_built_parcels.loc[df_built_parcels.Total_Remaining_Units > 0]
df_remaining.Reason.to_list() 

# get first text before first space
df_remaining['Jurisdiction'] = df_remaining['Reason'].str.split(' ').str[0]
df_remaining

,Reason,Parcels_Available,Parcels_Used,Total_Forecasted_Units,Total_Remaining_Units,Jurisdiction
3,DG Bonus Units MF,0,0,0.0,23.0,DG
8,WA Bonus Units SF,0,0,0.0,42.0,WA
9,WA Bonus Units SF,0,0,0.0,60.0,WA
12,CSLT General Units SF,97,97,97.0,78.0,CSLT
14,EL General Units MF,14,12,34.0,72.0,EL
17,PL General Units MF,35,34,125.0,25.0,PL
20,DG General Units MF,9,0,0.0,18.0,DG
23,WA General Units MF,19,10,29.0,28.0,WA
24,WA General Units SF,46,46,46.0,35.0,WA
26,TRPA Bonus Units MF,21,13,35.0,23.0,TRPA


In [37]:
##------------------------------------ Infill Forecasting remaining units from other Pools ------------------------------------##

# Forecast based on target sum of remaining units but using infill function and condition
target_sum = df_remaining.loc[df_remaining.Reason == 'WA Bonus Units SF', 'Total_Remaining_Units'].values[0]
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, WA_Bonus_infill_condition, target_sum, 'WA Bonus Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

target_sum = df_remaining.loc[df_remaining.Reason == 'EL General Units MF', 'Total_Remaining_Units'].values[0]
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, EL_infill_condition, target_sum, 'EL General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

target_sum = df_remaining.loc[df_remaining.Reason == 'PL General Units MF', 'Total_Remaining_Units'].values[0]
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, PL_infill_condition, target_sum, 'PL General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

target_sum = df_remaining.loc[df_remaining.Reason == 'DG General Units MF', 'Total_Remaining_Units'].values[0]
sdfParcels, df_summary = forecast_residential_units(sdfParcels, DG_infill_condition, target_sum, 'DG General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

target_sum = df_remaining.loc[df_remaining.Reason == 'WA General Units MF', 'Total_Remaining_Units'].values[0]
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, WA_infill_condition, target_sum, 'WA General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

target_sum = df_remaining.loc[df_remaining.Reason == 'WA General Units SF', 'Total_Remaining_Units'].values[0]
sdfParcel, df_summary = forecast_residential_units_infill(sdfParcels, WA_infill_condition, target_sum, 'WA General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

# TRPA General Remaining Units
target_sum = df_remaining.loc[df_remaining.Reason == 'TRPA General Units MF', 'Total_Remaining_Units'].values[0]
sdfParcels, df_summary = forecast_residential_units_infill(sdfParcels, TRPA_infill_condition, target_sum, 'TRPA General Units Infill')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

df_built_parcels

,Reason,Parcels_Available,Parcels_Used,Total_Forecasted_Units,Total_Remaining_Units
0,CSLT Bonus Units MF,53,10,24.0,0.0
1,CSLT Bonus Units SF,43,5,34.0,0.0
2,CSLT Bonus Units Infill,1993,2,10.0,0.0
3,DG Bonus Units MF,0,0,0.0,23.0
4,DG Bonus Units Infill,445,3,10.0,0.0
5,PL Bonus Units MF,214,10,10.0,0.0
6,PL Bonus Units SF,37,2,14.0,0.0
7,PL Bonus Units Infill,1140,2,4.0,0.0
8,WA Bonus Units SF,0,0,0.0,42.0
9,WA Bonus Units SF,0,0,0.0,60.0


In [38]:
# assign ADUs to existing residential parcels residential units=1 and ADU_ALLOWED = Yes
target_sum = 4385 - sdfParcels.FORECASTED_RESIDENTIAL_UNITS.sum()
sdfParcels, df_summary = forecast_residential_units(sdfParcels, TRPA_ADU_condition, target_sum, 'TRPA ADU Units')
df_built_parcels = pd.concat([df_built_parcels, df_summary], ignore_index=True)

> Check Results

In [39]:
dfPoolMelt = dfPool.melt(id_vars=['Jurisdiction', 'Unit_Pool'], value_vars=['Future_Units_Adjusted_MF', 'Future_Units_Adjusted_SF', 'Future_Units_Adjusted_Infill'])
# drop Future_Units_Adjusted_ from variable
dfPoolMelt['variable'] = dfPoolMelt['variable'].str.replace('Future_Units_Adjusted_', '')

dfPoolMelt['Unit_Pool'] = dfPoolMelt['Jurisdiction'] + ' ' + dfPoolMelt['Unit_Pool']
dfPoolMelt.rename(columns={'variable': 'Reason', 'value': 'Units'}, inplace=True)

dfForecastGroup = sdfParcels.groupby(['FORECAST_REASON'])['FORECASTED_RESIDENTIAL_UNITS'].sum().reset_index()
# split last FORECAST Reason into two columns based on last space in string
dfForecastGroup['Reason'] = dfForecastGroup['FORECAST_REASON'].str.split(' ').str[-1]
dfForecastGroup['Jurisdiction'] = dfForecastGroup['FORECAST_REASON'].str.split(' ').str[0]
# if FORECAST_REASON valie contains 'Bonus' then set Unit Pool to Bonus Units
dfForecastGroup.loc[dfForecastGroup['FORECAST_REASON'].str.contains('Bonus'), 'Unit_Pool'] = dfForecastGroup.Jurisdiction + ' ' + 'Bonus Unit'
# if FORECAST_REASON valie contains 'General' then set Unit Pool to General Units
dfForecastGroup.loc[dfForecastGroup['FORECAST_REASON'].str.contains('General'), 'Unit_Pool'] = dfForecastGroup.Jurisdiction + ' ' + 'General'
dfMerge = dfForecastGroup.merge(dfPoolMelt, on=['Unit_Pool', 'Reason'], how='left')
dfMerge['Unit_Diff'] = dfMerge['FORECASTED_RESIDENTIAL_UNITS'] - dfMerge['Units']
dfMerge

,FORECAST_REASON,FORECASTED_RESIDENTIAL_UNITS,Reason,Jurisdiction_x,Unit_Pool,Jurisdiction_y,Units,Unit_Diff
0,CSLT Bonus Units Infill,20,Infill,CSLT,CSLT Bonus Unit,CSLT,10.0,10.0
1,CSLT Bonus Units MF,48,MF,CSLT,CSLT Bonus Unit,CSLT,24.0,24.0
2,CSLT Bonus Units SF,68,SF,CSLT,CSLT Bonus Unit,CSLT,34.0,34.0
3,CSLT General Units Infill,52,Infill,CSLT,CSLT General,CSLT,52.0,0.0
4,CSLT General Units MF,244,MF,CSLT,CSLT General,CSLT,122.0,122.0
5,CSLT General Units SF,272,SF,CSLT,CSLT General,CSLT,175.0,97.0
6,CTC Asset Lands,272,Lands,CTC,NaN,NaN,NaN,NaN
7,DG Bonus Units Infill,20,Infill,DG,DG Bonus Unit,DG,10.0,10.0
8,DG Bonus Units MF,23,MF,DG,DG Bonus Unit,DG,23.0,0.0
9,DG General Units Infill,26,Infill,DG,DG General,DG,8.0,18.0


> Assign Forecast Year

In [40]:
## Assigning Development Year to Parcels ##
# Get total development by 2035
TotalDevelopment = dfResZoned.Future_Units.sum()
Development_2035 = (TotalDevelopment*.33).astype(int)
sdfParcels['Development_Year']=None
#Assining all development that's currently in the works as being cone by 2035
sdfParcels.loc[sdfParcels['FORECAST_REASON'] == 'Assigned', 'Development_Year'] = 2035
RemainingDevelopment_2035 = Development_2035 - sdfParcels.loc[sdfParcels['FORECAST_REASON'] == 'Assigned', 'FORECASTED_RESIDENTIAL_UNITS'].sum()

Development_2035_Condition = sdfParcels['FORECASTED_RESIDENTIAL_UNITS'].where(sdfParcels['FORECAST_REASON'] != 'Assigned').cumsum()
sdfParcels.loc[Development_2035_Condition < RemainingDevelopment_2035, 'Development_Year'] = 2035
sdfParcels.loc[(sdfParcels['FORECAST_REASON']!= '') & (sdfParcels['Development_Year'].isnull()), 'Development_Year'] = 2050
development_year = sdfParcels.groupby('Development_Year')['FORECASTED_RESIDENTIAL_UNITS'].sum()
development_year

Development_Year
2035    1446
2050    2939
Name: FORECASTED_RESIDENTIAL_UNITS, dtype: int64

#### Assign Occupancy Rate to all forecasted residential parcels

In [47]:
sdfParcels = pd.read_pickle(parcel_pickle_part2)
# filter to FORECAST_YEAR = 2035
#sdfParcels = sdfParcels.loc[sdfParcels['Development_Year'] == 2050]        

In [48]:
sdfParcels['FORECASTED_RES_OCCUPANCY_RATE'] = 0
# map lookup known project occupancy rates to parcels
string_cols = dfResAssigned.select_dtypes(include=['string']).columns
dfResAssigned[string_cols] = dfResAssigned[string_cols].astype(object)
dfResAssigned['Occupancy_Rate'] = dfResAssigned['Occupancy_Rate'].astype(float)
# change all strings in dfResAssigned to be objects
string_cols=sdfParcels.select_dtypes(include=['string']).columns
sdfParcels[string_cols] = sdfParcels[string_cols].astype(object)
sdfParcels['FORECASTED_RES_OCCUPANCY_RATE'] = sdfParcels['APN'].map(dict(zip(dfResAssigned.APN, dfResAssigned['Occupancy_Rate'])))
sdfParcels.FORECAST_REASON.value_counts()
# if FORECAST_REASON has the word 'Bonus' in it set occupancy rate to 100%
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('Bonus'), 'FORECASTED_RES_OCCUPANCY_RATE'] = 1
# if FORECAST_REASON has the word 'CTC' in it set occupancy rate to 100%
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('CTC'), 'FORECASTED_RES_OCCUPANCY_RATE'] = 1
# if FORECAST_REASON has the word 'General' and housing zoning is SF_only in it set occupancy rate to 35
sdfParcels.loc[(sdfParcels['FORECAST_REASON'].fillna('').str.contains('General')), 'FORECASTED_RES_OCCUPANCY_RATE'] = 0.35

#### Assing Household Income Category - Low, Medium, High

In [49]:
sdfParcels['FORECASTED_RES_INCOME_LOW']     = 0.0
sdfParcels['FORECASTED_RES_INCOME_MEDIUM']  = 0.0
sdfParcels['FORECASTED_RES_INCOME_HIGH']    = 0.0

# map lookup known project income levels to parcels
sdfParcels['FORECAST_RES_INCOME_CATEGORY']     = sdfParcels.APN.map(dict(zip(dfResAssigned.APN, dfResAssigned['HH Income Category'])))
# change 'Achievable' to 'Medium' and 'Affodaable to 'Low'
sdfParcels.loc[sdfParcels['FORECAST_RES_INCOME_CATEGORY'] == 'Achievable', 'FORECAST_RES_INCOME_CATEGORY'] = 'Medium'
sdfParcels.loc[sdfParcels['FORECAST_RES_INCOME_CATEGORY'] == 'Affordable', 'FORECAST_RES_INCOME_CATEGORY'] = 'Low'

# def function to if income category is low, medium, or high income field to 1
def income_category(df, category):
    df.loc[df['FORECAST_RES_INCOME_CATEGORY'] == category, 'FORECASTED_RES_INCOME_' + str(category).upper()] = 1
    return df

for category in ['Low', 'Medium', 'High']:
    sdfParcels = income_category(sdfParcels, category)

# for FORECAST_REASON with 'Bonus' set low income to 1
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('Bonus'), 'FORECASTED_RES_INCOME_LOW'] = 0.78
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('Bonus'), 'FORECASTED_RES_INCOME_MEDIUM'] = 0.2
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('Bonus'), 'FORECASTED_RES_INCOME_HIGH'] = 0.02

# for FORECAST_REASON with 'General' and housing zoning is SF_only set low income to 1
sdfParcels.loc[(sdfParcels['FORECAST_REASON'].fillna('').str.contains('General')), 'FORECASTED_RES_INCOME_LOW'] = 0.01
sdfParcels.loc[(sdfParcels['FORECAST_REASON'].fillna('').str.contains('General')), 'FORECASTED_RES_INCOME_MEDIUM'] = 0.02
sdfParcels.loc[(sdfParcels['FORECAST_REASON'].fillna('').str.contains('General')), 'FORECASTED_RES_INCOME_HIGH'] = 0.97

# assigne income levels to parcels with 'ADU' in FORECAST_REASON - UPDATED
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('ADU'), 'FORECASTED_RES_INCOME_LOW'] = 0.65
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('ADU'), 'FORECASTED_RES_INCOME_MEDIUM'] = 0.2
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('ADU'), 'FORECASTED_RES_INCOME_HIGH'] = 0.15

# for FORECAST_REASON with 'CTC' set low income to 1
sdfParcels.loc[sdfParcels['FORECAST_REASON'].fillna('').str.contains('CTC'), 'FORECASTED_RES_INCOME_LOW'] = 1



### Summary

In [ ]:
# Need to get 2035 forecasted units and 2050 forecasted units
# We need to also change the people per household to hit population targets

In [50]:
# bring in the TAZ s
socio            = 'TravelDemandModel\\2022\\data\\processed_data\\SocioEcon_Summer.csv'
socio_path       = local_path.parents[2].joinpath(socio)
dfSocio          = pd.read_csv(socio_path)

# rename columns to taz to TAZ
dfSocio.rename(columns={'taz': 'TAZ'}, inplace=True)
# group by TAZ
sdfParcels['FORECASTED_OCCUPIED_UNITS'] = sdfParcels['FORECASTED_RESIDENTIAL_UNITS'] * sdfParcels['FORECASTED_RES_OCCUPANCY_RATE']
sdfParcels['FORECASTED_RES_INCOME_LOW_UNITS']    = sdfParcels['FORECASTED_RES_INCOME_LOW'] * sdfParcels['FORECASTED_OCCUPIED_UNITS']
sdfParcels['FORECASTED_RES_INCOME_MEDIUM_UNITS'] = sdfParcels['FORECASTED_RES_INCOME_MEDIUM'] * sdfParcels['FORECASTED_OCCUPIED_UNITS']
sdfParcels['FORECASTED_RES_INCOME_HIGH_UNITS']   = sdfParcels['FORECASTED_RES_INCOME_HIGH'] * sdfParcels['FORECASTED_OCCUPIED_UNITS']

df = sdfParcels.loc[sdfParcels['FORECASTED_RESIDENTIAL_UNITS'] > 0]
dfTAZ_Summary_2035 = df.loc[df['Development_Year'] == 2035]
dfTAZ_Summary_2035 = dfTAZ_Summary_2035.groupby(['TAZ']).agg({'FORECASTED_RESIDENTIAL_UNITS': 'sum',
                                                                                    'FORECASTED_OCCUPIED_UNITS':'sum', 
                                                                                    'FORECASTED_RES_INCOME_LOW_UNITS':'sum', 'FORECASTED_RES_INCOME_MEDIUM_UNITS':'sum',
                                                                                    'FORECASTED_RES_INCOME_HIGH_UNITS':'sum'}).reset_index()

#This is total development by 2050
dfTAZ_Summary_2050 = df.groupby(['TAZ']).agg({'FORECASTED_RESIDENTIAL_UNITS': 'sum',
                                                 'FORECASTED_OCCUPIED_UNITS':'sum', 
                                                 'FORECASTED_RES_INCOME_LOW_UNITS':'sum', 'FORECASTED_RES_INCOME_MEDIUM_UNITS':'sum',
                                                 'FORECASTED_RES_INCOME_HIGH_UNITS':'sum'}).reset_index()

# merge TAZ summary with socio data

def process_dataframe(df, dfSocio):
    df = dfSocio.merge(df, on='TAZ', how='left')
    df = df.fillna(0)
    
    df['TOTAL_FORECASTED_UNITS_LOW_INCOME'] = df['FORECASTED_RES_INCOME_LOW_UNITS'] + df['occ_units_low_inc']
    df['TOTAL_FORECASTED_UNITS_MED_INCOME'] = df['FORECASTED_RES_INCOME_MEDIUM_UNITS'] + df['occ_units_med_inc']
    df['TOTAL_FORECASTED_UNITS_HIGH_INCOME'] = df['FORECASTED_RES_INCOME_HIGH_UNITS'] + df['occ_units_high_inc']
    df['TOTAL_FORECASTED_UNITS_OCCUPIED'] = df['FORECASTED_OCCUPIED_UNITS'] + df['total_occ_units']
    df['NEW_OCCUPANCY_RATE'] = (df['FORECASTED_OCCUPIED_UNITS'] + df['total_occ_units']) / (df['total_residential_units'] + df['FORECASTED_RESIDENTIAL_UNITS'])
    
    return df
dfTAZ_Summary_2035 = process_dataframe(dfTAZ_Summary_2035, dfSocio)
dfTAZ_Summary_2050 = process_dataframe(dfTAZ_Summary_2050, dfSocio)

# dfTAZ_Summary = dfTAZ_Summary.merge(dfSocio, on='TAZ', how='left')

# dfTAZ_Summary['TOTAL_FORECASTED_UNITS_LOW_INCOME'] = dfTAZ_Summary['FORECASTED_RES_INCOME_LOW_UNITS'] + dfTAZ_Summary['occ_units_low_inc']
# dfTAZ_Summary['TOTAL_FORECASTED_UNITS_MED_INCOME'] = dfTAZ_Summary['FORECASTED_RES_INCOME_MEDIUM_UNITS'] + dfTAZ_Summary['occ_units_med_inc']
# dfTAZ_Summary['TOTAL_FORECASTED_UNITS_HIGH_INCOME'] = dfTAZ_Summary['FORECASTED_RES_INCOME_HIGH_UNITS'] + dfTAZ_Summary['occ_units_high_inc']
# dfTAZ_Summary['NEW_OCCUPANCY_RATE'] = (dfTAZ_Summary['FORECASTED_OCCUPIED_UNITS'] + dfTAZ_Summary['total_occ_units'])/ (dfTAZ_Summary['total_residential_units'] + dfTAZ_Summary['FORECASTED_RESIDENTIAL_UNITS'])

In [51]:
target_sum_2035 = 55592
target_sum_2050 =57611

dfTAZ_Summary_2035['FORECASTED_POPULATION'] = dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_OCCUPIED']*dfTAZ_Summary_2035['persons_per_occ_unit']
forecasted_population_2035 = dfTAZ_Summary_2035['FORECASTED_POPULATION'].sum()
dfTAZ_Summary_2050['FORECASTED_POPULATION'] = dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_OCCUPIED']*dfTAZ_Summary_2050['persons_per_occ_unit']
forecasted_population_2050 = dfTAZ_Summary_2050['FORECASTED_POPULATION'].sum()
adjustment_factor_2035 = target_sum_2035/forecasted_population_2035
adjustment_factor_2050  = target_sum_2050/forecasted_population_2050

dfTAZ_Summary_2035['Adjusted_Persons_Per_Occupied_Unit'] = dfTAZ_Summary_2035['persons_per_occ_unit']*adjustment_factor_2035
dfTAZ_Summary_2050['Adjusted_Persons_Per_Occupied_Unit'] = dfTAZ_Summary_2050['persons_per_occ_unit']*adjustment_factor_2050

dfTAZ_Summary_2050['FORECASTED_POPULATION'] = dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_OCCUPIED']*dfTAZ_Summary_2050['Adjusted_Persons_Per_Occupied_Unit']
dfTAZ_Summary_2035['FORECASTED_POPULATION'] = dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_OCCUPIED']*dfTAZ_Summary_2035['Adjusted_Persons_Per_Occupied_Unit']
forecasted_population_2035 = dfTAZ_Summary_2035['FORECASTED_POPULATION'].sum()
forecasted_population_2050 = dfTAZ_Summary_2050['FORECASTED_POPULATION'].sum()


In [ ]:
print(forecasted_population_2035)
print(forecasted_population_2050)




In [ ]:
print(dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_OCCUPIED'].sum())
print(dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_OCCUPIED'].sum())

In [ ]:
print(dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_MED_INCOME'].sum()+dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_LOW_INCOME'].sum()+dfTAZ_Summary_2035['TOTAL_FORECASTED_UNITS_HIGH_INCOME'].sum())
print(dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_MED_INCOME'].sum()+dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_LOW_INCOME'].sum()+dfTAZ_Summary_2050['TOTAL_FORECASTED_UNITS_HIGH_INCOME'].sum())

In [52]:
# save to pickle
taz_summary_2035_pickle = data_dir / 'taz_summary_2035.pickle'
taz_summary_2050_pickle = data_dir / 'taz_summary_2050.pickle'
dfTAZ_Summary_2035.to_pickle(taz_summary_2035_pickle)
dfTAZ_Summary_2035.to_csv(data_dir / 'taz_summary_2035.csv')
dfTAZ_Summary_2050.to_pickle(taz_summary_2050_pickle)
dfTAZ_Summary_2050.to_csv(data_dir / 'taz_summary_2050.csv')

##### QA Extra

In [53]:
built_units = sdfParcels.groupby('FORECAST_REASON').agg({'FORECASTED_RESIDENTIAL_UNITS':'sum'})
dfResZoned = dfPool.copy()
# Create a dictionary to map the forecast reason to the Jurisdiction and Unit Pool
Forecast_Reason_lookup = {'CSLT Bonus Units Built':{'Jurisdiction':'CSLT', 'Unit_Pool':'Bonus Unit'},
                          'CSLT General Units Built':{'Jurisdiction':'CSLT', 'Unit_Pool':'General'},
                          'EL General Units Built':{'Jurisdiction':'EL', 'Unit_Pool':'General'},
                          'PL Bonus Units Built':{'Jurisdiction':'PL', 'Unit_Pool':'Bonus Unit'},
                          'PL General Units Built':{'Jurisdiction':'PL', 'Unit_Pool':'General'},
                          'WA Bonus Units Built':{'Jurisdiction':'WA', 'Unit_Pool':'Bonus Unit'},
                          'WA General Units Built':{'Jurisdiction':'WA', 'Unit_Pool':'General'},
                          'DG Bonus Units Built':{'Jurisdiction':'DG', 'Unit_Pool':'Bonus Unit'},
                          'DG General Units Built':{'Jurisdiction':'DG', 'Unit_Pool':'General'},
                          'TRPA Bonus Units Built':{'Jurisdiction':'TRPA', 'Unit_Pool':'Bonus Unit'},
                          'TRPA General Units Built':{'Jurisdiction':'TRPA', 'Unit_Pool':'General'},
                          'ADU Units Built':{'Jurisdiction':'TRPA', 'Unit_Pool':'ADU'}}
# Map 'Jurisdiction' and 'Unit_Pool' separately from the dictionary
built_units['Jurisdiction'] = built_units.index.map(lambda x: Forecast_Reason_lookup.get(x, {}).get('Jurisdiction'))
built_units['Unit_Pool'] = built_units.index.map(lambda x: Forecast_Reason_lookup.get(x, {}).get('Unit_Pool'))
unit_comparison = built_units.merge(dfResZoned, how='left', on=['Jurisdiction', 'Unit_Pool'])
unit_comparison['Difference'] = unit_comparison['Future_Units_Adjusted'] - unit_comparison['FORECASTED_RESIDENTIAL_UNITS']
unit_comparison.to_csv('unit_comparison.csv', index=False)

In [54]:
# forecasted residential uints by location to twon center
sdfParcels.groupby(['LOCATION_TO_TOWNCENTER','FORECAST_REASON'])['FORECASTED_RESIDENTIAL_UNITS'].sum()

LOCATION_TO_TOWNCENTER                      FORECAST_REASON          
Further than Quarter Mile from Town Center  Assigned                     482.0
                                            CSLT Bonus Units MF            4.0
                                            CSLT Bonus Units SF            1.0
                                            CSLT General Units Infill     17.0
                                            CSLT General Units MF         29.0
                                            CSLT General Units SF        114.0
                                            CTC Asset Lands               13.0
                                            DG Bonus Units Infill          2.0
                                            DG Bonus Units MF              1.0
                                            DG General Units Infill       14.0
                                            DG General Units SF           21.0
                                            EL General Units 

In [55]:
# gropu by TAZ and sum forecasted residential units and residential units
# Forecast year total residential units by TAZ
sdfParcels.groupby('TAZ')[['FORECASTED_RESIDENTIAL_UNITS', 'Residential_Units']].sum().reset_index()
# total units with forecast year 2035 and 2050
sdfParcels.groupby('Development_Year')[['FORECASTED_RESIDENTIAL_UNITS', 'Residential_Units']].sum().reset_index()

,Development_Year,FORECASTED_RESIDENTIAL_UNITS,Residential_Units
0,2035,2003.0,484
1,2050,2382.0,48940


### Tourist Accommodation Forecast

In [56]:
# lookup lists
tau_assigned_lookup = "Lookup_Lists/forecast_tourist_assigned_units.csv"
# get assigned units lookup as data frames
dfTouristAssigned = pd.read_csv(tau_assigned_lookup)
# assign tourist units to parcels
sdfParcels['FORECASTED_TOURIST_UNITS'] = 0
# set tourist units to assigned total
sdfParcels['FORECASTED_TOURIST_UNITS'] = sdfParcels.APN.map(dict(zip(dfTouristAssigned.APN, dfTouristAssigned['Unit_Pool'])))

### Commercial Floor Area Forecast

In [58]:
# lookup lists
cfa_assigned_lookup = "Lookup_Lists/forecast_commercial_assigned_units.csv"
# get zoned units lookups as data frames
dfCommercialAssigned = pd.read_csv(cfa_assigned_lookup)
# set commercial floor area to assigned total
sdfParcels['FORECASTED_COMMERCIAL_FLOOR_AREA'] = 0
sdfParcels['FORECASTED_COMMERCIAL_FLOOR_AREA'] = sdfParcels.APN.map(dict(zip(dfCommercialAssigned.APN, dfCommercialAssigned.SqFt)))

### Exports

In [59]:
# export to pickle
sdfParcels.to_pickle(parcel_pickle_part2)
# to csv
sdfParcels.to_csv(data_dir/'Parcels_Forecast.csv', index=False)
# to feature class
sdfParcels.spatial.to_featureclass(Path(gdb)/'Parcel_Forecast')

# Summarize Existing and Forecasted Units by Jurisdiction and Unit Pool by TAZ
dfTAZ = sdfParcels.groupby('TAZ')[['FORECASTED_RESIDENTIAL_UNITS', 'Residential_Units']].sum().reset_index()
dfTAZ.to_csv(data_dir/'TAZ_Units.csv', index=False)